# AML Transaction Monitoring & SAR Filing - Interactive Walkthrough

## Overview & Regulatory Context

**Regulation**: BSA/AML / FinCEN Suspicious Activity Report (SAR) Requirements  
**Regulators**: FinCEN, OCC, Federal Reserve, FDIC  
**Key Challenge**: Rule or threshold changes create retroactive examination liability

### Critical Compliance Problem
When AML monitoring rules change mid-year, banks face a significant regulatory risk:
- **Retroactive Review**: Examiners may question why similar transactions received different treatment before and after rule changes
- **Threshold Accountability**: Banks must prove which thresholds were in effect for each alert and SAR filing
- **MRA Protection**: Rule version tracking prevents "Matter Requiring Attention" citations

### Learning Objectives
1. Track AML alert generation with complete rule version history
2. Capture threshold-in-effect for each transaction decision
3. Create immutable audit trails for SAR filing decisions
4. Demonstrate rule change impact analysis for examiner queries

## Setup & SDK Initialization

Let's begin by importing required modules and initializing the Briefcase AI SDK:

In [ ]:
import sys
import os
import uuid
import random
from datetime import datetime, timedelta
from typing import Dict, Any

# Add shared module to path
_p = os.path.abspath('')
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, 'shared')):
    _p = os.path.dirname(_p)
if os.path.isdir(os.path.join(_p, 'shared')):
    sys.path.insert(0, os.path.join(_p, 'shared'))

try:
    import backend
    from backend import briefcase, DecisionSnapshot, Input, Output, SqliteBackend
    print("[SUCCESS] Successfully imported Briefcase AI SDK and backend utilities")
except ImportError as e:
    print(f"[FAILED] Error importing required modules: {e}")
    print("Please ensure the shared backend module is available")

In [ ]:
# Initialize Briefcase AI SDK
try:
    briefcase.init_with_config(2)  # Initialize with 2 worker threads
    print("[SUCCESS] Briefcase AI SDK initialized successfully")
    
    # Get configured backend for audit trail storage
    db_backend = backend.get_backend()
    print("[SUCCESS] SQLite backend configured for immutable audit storage")
    
except Exception as e:
    print(f"[FAILED] Failed to initialize SDK: {e}")

## AML Rule Configuration (Version 4.1)

We'll start with the original AML rule configuration, then simulate a mid-year threshold change:

In [ ]:
# AML Rule Configuration v4.1 (before threshold change)
aml_rules_v41 = {
    "version": "aml-rules-v4.1",
    "alert_thresholds": {
        "large_cash_transaction": 0.5,      # CTR threshold-related alerts
        "structuring_pattern": 0.6,         # Potential structuring detection
        "wire_to_high_risk_country": 0.7    # International wire monitoring
    },
    "effective_date": datetime.utcnow() - timedelta(days=60)
}

print("[CONFIG] Initial AML Rule Configuration (v4.1):")
print(f"   Version: {aml_rules_v41['version']}")
print(f"   Effective Date: {aml_rules_v41['effective_date'].strftime('%Y-%m-%d')}")
print("   Alert Thresholds:")
for rule_id, threshold in aml_rules_v41["alert_thresholds"].items():
    print(f"     • {rule_id}: {threshold}")
    
print("\n**Details:** Regulatory Context:")
print("   • Large Cash: $10K+ transactions (BSA CTR requirements)")
print("   • Structuring: Multiple transactions <$10K (31 USC 5324)")
print("   • High-Risk Countries: OFAC/FinCEN geographic risk lists")

## AML Alert Scoring Engine

This function simulates how banks generate AML alerts based on transaction patterns and rule thresholds:

In [ ]:
def simulate_aml_alert_scoring(transaction_data: Dict[str, Any], rule_config: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates AML alert generation and scoring based on transaction patterns.
    In production, this would be complex ML models and rules engines.
    """
    transaction_amount = transaction_data["transaction_amount"]
    transaction_type = transaction_data["transaction_type"]
    alert_rule_id = transaction_data["alert_rule_id"]
    
    # Base alert score calculation
    alert_score = 0.0
    
    # Amount-based scoring logic
    if alert_rule_id == "large_cash_transaction":
        # $10K+ cash transactions (CTR threshold)
        if transaction_amount >= 10000:
            alert_score = min(1.0, transaction_amount / 50000)
    
    elif alert_rule_id == "structuring_pattern":
        # Multiple transactions just under $10K (structuring detection)
        if transaction_amount >= 9000 and transaction_amount < 10000:
            alert_score = 0.7 + random.uniform(0.0, 0.2)
    
    elif alert_rule_id == "wire_to_high_risk_country":
        # International wires to high-risk countries
        if transaction_type == "wire" and transaction_amount > 5000:
            alert_score = 0.6 + random.uniform(0.0, 0.3)
    
    # Add slight randomness to simulate complex rule interactions
    alert_score += random.uniform(-0.05, 0.1)
    alert_score = max(0.0, min(1.0, alert_score))
    
    # Apply threshold from current rule configuration
    threshold = rule_config["alert_thresholds"][alert_rule_id]
    
    # Generate alert if score exceeds threshold
    alert_generated = alert_score >= threshold
    
    return {
        "alert_generated": alert_generated,
        "alert_score": round(alert_score, 3),
        "threshold_at_alert_time": threshold,
        "alert_rule_version": rule_config["version"]
    }

print("[SUCCESS] AML Alert Scoring Engine defined")
print("   **Objective:** Captures rule version and threshold at decision time")
print("   **Results:** Provides explainable scoring for analyst review")
print("   [SECURED] Creates immutable audit trail for each alert")

## Analyst Review Simulation

This simulates how analysts review AML alerts and make SAR filing decisions:

In [ ]:
def simulate_analyst_review(alert_data: Dict[str, Any], transaction_data: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simulates analyst review of an AML alert and SAR filing decision.
    """
    alert_score = alert_data["alert_score"]
    transaction_amount = transaction_data["transaction_amount"]
    
    # Simulate analyst decision-making process
    if alert_score >= 0.8 and transaction_amount >= 25000:
        # High-risk, large amount → file SAR
        decision = "file_sar"
        sar_filed = True
        sar_id = f"SAR-{datetime.utcnow().strftime('%Y%m%d')}-{random.randint(1000, 9999)}"
        rationale = f"High-risk transaction: Score {alert_score}, Amount ${transaction_amount:,.0f}"
        
    elif alert_score >= 0.5:
        # Medium risk → additional monitoring but clear for now
        decision = "clear"
        sar_filed = False
        sar_id = None
        rationale = f"Medium risk cleared: Score {alert_score}, monitoring continues"
        
    else:
        # Low risk → clear
        decision = "clear"
        sar_filed = False
        sar_id = None
        rationale = f"Low risk cleared: Score {alert_score}"
    
    return {
        "analyst_decision": decision,
        "sar_filed": sar_filed,
        "sar_id": sar_id,
        "decision_rationale": rationale,
        "decision_timestamp": datetime.utcnow().isoformat(),
        "analyst_id": "analyst_" + str(random.randint(100, 999))
    }

print("[SUCCESS] Analyst Review System defined")
print("   👤 Captures analyst ID and decision rationale")
print("   📄 Tracks SAR filing decisions with unique IDs")
print("   ⏰ Records precise decision timestamps")

## Transaction Processing (Rules v4.1)

Let's process our first transaction using the original rule configuration:

In [ ]:
print("="*70)
print("TRANSACTION MONITORING (Rules v4.1)")
print("="*70)

# Create first transaction for processing
alert_id_1 = str(uuid.uuid4())
transaction_id_1 = str(uuid.uuid4())

transaction_data_1 = {
    "alert_id": alert_id_1,
    "transaction_id": transaction_id_1,
    "customer_id": str(uuid.uuid4()),
    "transaction_amount": 27500.0,
    "transaction_type": "wire",
    "alert_rule_id": "wire_to_high_risk_country",
    "alert_rule_version": aml_rules_v41["version"],
    "analyst_id": "analyst_205"
}

print("**Analysis:** Processing High-Value Wire Transaction:")
print(f"   Transaction ID: {transaction_id_1[:12]}...")
print(f"   Amount: ${transaction_data_1['transaction_amount']:,.0f}")
print(f"   Type: {transaction_data_1['transaction_type']}")
print(f"   Rule: {transaction_data_1['alert_rule_id']}")
print(f"   Rule Version: {aml_rules_v41['version']}")

In [ ]:
# Generate alert using rules v4.1
alert_result_1 = simulate_aml_alert_scoring(transaction_data_1, aml_rules_v41)

print("**Results:** Alert Scoring Results:")
print(f"   Alert Score: {alert_result_1['alert_score']}")
print(f"   Threshold (v4.1): {alert_result_1['threshold_at_alert_time']}")
print(f"   Alert Generated: {'[SUCCESS] YES' if alert_result_1['alert_generated'] else '[FAILED] NO'}")

if alert_result_1["alert_generated"]:
    print("\n👤 Routing to Analyst for Review...")
    
    # Analyst review process
    analyst_result_1 = simulate_analyst_review(alert_result_1, transaction_data_1)
    
    print(f"   Analyst Decision: {analyst_result_1['analyst_decision'].upper()}")
    print(f"   Rationale: {analyst_result_1['decision_rationale']}")
    
    if analyst_result_1["sar_filed"]:
        print(f"   📄 SAR Filed: {analyst_result_1['sar_id']}")
        print(f"   ⏰ Filing Time: {analyst_result_1['decision_timestamp']}")
else:
    print("\n[SUCCESS] No alert generated - below threshold")

## Audit Trail Creation (Transaction 1)

Create an immutable audit trail capturing the complete decision context:

In [ ]:
if alert_result_1["alert_generated"]:
    # Create regulatory metadata for audit trail
    regulatory_metadata_1 = {
        "regulation": "BSA/AML / FinCEN SAR Requirements",
        "rule_version_history_preserved": True,
        "mra_protection": True,  # Protects against Matter Requiring Attention citations
        "threshold_at_decision_time": alert_result_1["threshold_at_alert_time"]
    }
    
    print("💾 Creating Immutable Audit Trail...")
    print(f"   Rule Version: {alert_result_1['alert_rule_version']}")
    print(f"   Threshold Captured: {regulatory_metadata_1['threshold_at_decision_time']}")
    print(f"   MRA Protection: {regulatory_metadata_1['mra_protection']}")
    
    # Create DecisionSnapshot
    snapshot_1 = backend.create_decision_snapshot(
        function_name="aml_alert_analysis",
        inputs=transaction_data_1,
        outputs={**alert_result_1, **analyst_result_1},
        metadata=regulatory_metadata_1
    )
    
    # Store in immutable audit system
    stored_id_1 = db_backend.save_decision(snapshot_1)
    
    print(f"[SUCCESS] Audit Trail Stored: {stored_id_1[:12]}...")
    print("   **Details:** Complete decision context preserved")
    print("   [SECURED] Immutable storage ensures integrity")
    print("   ⚖ Regulatory examination ready")

## Mid-Year Rule Change Simulation

Now we'll simulate a common regulatory challenge - rule threshold changes mid-year:

In [ ]:
print("\n" + "="*70)
print("[ALERT] MID-YEAR RULE CHANGE (Rules v4.2)")
print("="*70)

# AML Rule Configuration v4.2 (after threshold change)
aml_rules_v42 = {
    "version": "aml-rules-v4.2",
    "alert_thresholds": {
        "large_cash_transaction": 0.4,      # Lowered (more sensitive)
        "structuring_pattern": 0.5,         # Lowered
        "wire_to_high_risk_country": 0.6    # Lowered
    },
    "effective_date": datetime.utcnow()
}

print("**Details:** Rule Configuration Updated:")
print(f"   New Version: {aml_rules_v42['version']}")
print(f"   Effective Date: {aml_rules_v42['effective_date'].strftime('%Y-%m-%d %H:%M:%S')}")
print("\n**Results:** Threshold Changes (Impact Analysis):")

for rule_id in aml_rules_v41["alert_thresholds"]:
    old_threshold = aml_rules_v41["alert_thresholds"][rule_id]
    new_threshold = aml_rules_v42["alert_thresholds"][rule_id]
    change = new_threshold - old_threshold
    
    print(f"   • {rule_id}:")
    print(f"     Old: {old_threshold} → New: {new_threshold} ({change:+.1f})")
    print(f"     Impact: {'More sensitive' if change < 0 else 'Less sensitive'}")

print("\n**Objective:** Regulatory Risk: Lower thresholds may generate more alerts")
print("   Examiners will expect consistent treatment of similar transactions")

## Transaction Processing (Rules v4.2)

Process a similar transaction under the new rules to show the impact:

In [ ]:
# Create second transaction for processing under new rules
alert_id_2 = str(uuid.uuid4())
transaction_id_2 = str(uuid.uuid4())

transaction_data_2 = {
    "alert_id": alert_id_2,
    "transaction_id": transaction_id_2,
    "customer_id": str(uuid.uuid4()),
    "transaction_amount": 15000.0,  # Lower amount than first transaction
    "transaction_type": "wire",
    "alert_rule_id": "wire_to_high_risk_country",
    "alert_rule_version": aml_rules_v42["version"],
    "analyst_id": "analyst_207"
}

print("**Analysis:** Processing Wire Transaction Under New Rules:")
print(f"   Transaction ID: {transaction_id_2[:12]}...")
print(f"   Amount: ${transaction_data_2['transaction_amount']:,.0f} (lower than first)")
print(f"   Rule Version: {aml_rules_v42['version']}")
print(f"   New Threshold: {aml_rules_v42['alert_thresholds']['wire_to_high_risk_country']}")

# Generate alert using rules v4.2
alert_result_2 = simulate_aml_alert_scoring(transaction_data_2, aml_rules_v42)

print(f"\n**Results:** Alert Scoring Results:")
print(f"   Alert Score: {alert_result_2['alert_score']}")
print(f"   Threshold (v4.2): {alert_result_2['threshold_at_alert_time']}")
print(f"   Alert Generated: {'[SUCCESS] YES' if alert_result_2['alert_generated'] else '[FAILED] NO'}")

if alert_result_2["alert_generated"]:
    print("\n👤 Routing to Analyst for Review...")
    
    analyst_result_2 = simulate_analyst_review(alert_result_2, transaction_data_2)
    
    print(f"   Analyst Decision: {analyst_result_2['analyst_decision'].upper()}")
    print(f"   Rationale: {analyst_result_2['decision_rationale']}")
    
    if analyst_result_2.get("sar_filed"):
        print(f"   📄 SAR Filed: {analyst_result_2['sar_id']}")
else:
    print("\n[SUCCESS] No alert generated - still below threshold")

## Audit Trail Creation (Transaction 2)

Create audit trail for the second transaction with the new rule version:

In [ ]:
if alert_result_2["alert_generated"]:
    # Create regulatory metadata with new rule version
    regulatory_metadata_2 = {
        "regulation": "BSA/AML / FinCEN SAR Requirements",
        "rule_version_history_preserved": True,
        "mra_protection": True,
        "threshold_at_decision_time": alert_result_2["threshold_at_alert_time"]
    }
    
    print("💾 Creating Audit Trail for New Rule Version...")
    
    # Create DecisionSnapshot
    snapshot_2 = backend.create_decision_snapshot(
        function_name="aml_alert_analysis",
        inputs=transaction_data_2,
        outputs={**alert_result_2, **analyst_result_2},
        metadata=regulatory_metadata_2
    )
    
    # Store in immutable audit system
    stored_id_2 = db_backend.save_decision(snapshot_2)
    
    print(f"[SUCCESS] Audit Trail Stored: {stored_id_2[:12]}...")
    print(f"   **Details:** Rule Version: {alert_result_2['alert_rule_version']}")
    print(f"   **Results:** Threshold Captured: {regulatory_metadata_2['threshold_at_decision_time']}")
else:
    print("💾 Creating Audit Trail for No-Alert Decision...")
    # Still create audit trail even when no alert generated
    regulatory_metadata_2 = {
        "regulation": "BSA/AML / FinCEN SAR Requirements",
        "rule_version_history_preserved": True,
        "mra_protection": True,
        "threshold_at_decision_time": alert_result_2["threshold_at_alert_time"]
    }
    
    snapshot_2 = backend.create_decision_snapshot(
        function_name="aml_alert_analysis",
        inputs=transaction_data_2,
        outputs=alert_result_2,
        metadata=regulatory_metadata_2
    )
    
    stored_id_2 = db_backend.save_decision(snapshot_2)
    print(f"[SUCCESS] No-Alert Decision Stored: {stored_id_2[:12]}...")

## OCC/FinCEN Examiner Query Simulation

Demonstrate how to respond to examiner questions about rule versions and thresholds:

In [ ]:
print("\n" + "="*70)
print("👨‍**Business:** OCC/FINCEN EXAMINER SIMULATION")
print("="*70)

if 'stored_id_1' in locals() and alert_result_1.get("alert_generated") and 'analyst_result_1' in locals():
    sar_id = analyst_result_1.get("sar_id", "SAR-EXAMPLE-123")
    
    examiner_query = (
        f"What was the AML threshold in effect for rule "
        f"{transaction_data_1['alert_rule_id']} at the time SAR {sar_id} was filed? "
        f"How do you justify this threshold given current rule version {aml_rules_v42['version']}?"
    )
    
    print(f"**Details:** EXAMINER QUERY:")
    print(f"   {examiner_query}")
    print()
    
    # Generate response using immutable audit trail
    examiner_response = backend.format_examiner_response(
        stored_id_1,
        examiner_query,
        db_backend
    )
    
    print("**Bank:** BANK RESPONSE (From Immutable Audit Trail):")
    print(examiner_response)
    
else:
    print("**Details:** EXAMINER QUERY SIMULATION:")
    print("   (No SAR filed in this example, but audit trail still available)")
    print("   Example query: 'Show threshold in effect for each transaction'")

## Threshold Verification & Rule History

Demonstrate detailed threshold verification capabilities:

In [ ]:
if 'stored_id_1' in locals():
    print("**Analysis:** DETAILED THRESHOLD VERIFICATION:")
    
    # Retrieve and analyze first transaction
    retrieved_decision_1 = db_backend.load_decision(stored_id_1)
    if retrieved_decision_1:
        threshold_at_decision = retrieved_decision_1.tags.get("threshold_at_decision_time")
        rule_version = None
        
        # Extract rule version from inputs
        for inp in retrieved_decision_1.inputs:
            if inp.name == "alert_rule_version":
                rule_version = inp.value
                break
        
        print(f"   **Results:** Transaction 1 (High-value wire):")
        print(f"      Rule Version at Decision: {rule_version}")
        print(f"      Threshold in Effect: {threshold_at_decision}")
        print(f"      Current Threshold (v4.2): {aml_rules_v42['alert_thresholds']['wire_to_high_risk_country']}")
        print(f"      Threshold Change Impact: {float(threshold_at_decision) - aml_rules_v42['alert_thresholds']['wire_to_high_risk_country']:+.1f}")

if 'stored_id_2' in locals():
    # Retrieve and analyze second transaction
    retrieved_decision_2 = db_backend.load_decision(stored_id_2)
    if retrieved_decision_2:
        threshold_at_decision_2 = retrieved_decision_2.tags.get("threshold_at_decision_time")
        rule_version_2 = None
        
        for inp in retrieved_decision_2.inputs:
            if inp.name == "alert_rule_version":
                rule_version_2 = inp.value
                break
        
        print(f"   **Results:** Transaction 2 (Lower-value wire):")
        print(f"      Rule Version at Decision: {rule_version_2}")
        print(f"      Threshold in Effect: {threshold_at_decision_2}")
        
print("\n**Objective:** Regulatory Compliance Benefits:")
print("   [SUCCESS] Complete rule version history preserved")
print("   [SUCCESS] Threshold-in-effect documented for each decision")
print("   [SUCCESS] Retroactive examination queries answerable")
print("   [SUCCESS] MRA protection through audit completeness")

## Rule Version Impact Analysis

Analyze how threshold changes affected alert generation patterns:

In [ ]:
print("="*70)
print("**Metrics:** RULE VERSION IMPACT ANALYSIS")
print("="*70)

print("🔄 IMPACT OF THRESHOLD CHANGES:")

if 'alert_result_1' in locals():
    print(f"\n**Results:** Transaction 1 (Rules v4.1):")
    print(f"   Amount: ${transaction_data_1['transaction_amount']:,.0f}")
    print(f"   Alert Score: {alert_result_1['alert_score']}")
    print(f"   Threshold: {alert_result_1['threshold_at_alert_time']}")
    print(f"   Alert Generated: {'[SUCCESS] YES' if alert_result_1['alert_generated'] else '[FAILED] NO'}")
    if 'analyst_result_1' in locals() and analyst_result_1.get('sar_filed'):
        print(f"   SAR Filed: [SUCCESS] YES ({analyst_result_1['sar_id']})")
    else:
        print(f"   SAR Filed: [FAILED] NO")

if 'alert_result_2' in locals():
    print(f"\n**Results:** Transaction 2 (Rules v4.2):")
    print(f"   Amount: ${transaction_data_2['transaction_amount']:,.0f} (45% lower)")
    print(f"   Alert Score: {alert_result_2['alert_score']}")
    print(f"   Threshold: {alert_result_2['threshold_at_alert_time']} (14% lower)")
    print(f"   Alert Generated: {'[SUCCESS] YES' if alert_result_2['alert_generated'] else '[FAILED] NO'}")
    
    # Compare treatment of similar transactions
    print(f"\n⚖ CONSISTENCY ANALYSIS:")
    print(f"   • Lower transaction amount with lower threshold")
    print(f"   • Rule version changes properly documented")
    print(f"   • Different treatment justified by rule evolution")
    print(f"   • Audit trail supports examiner queries")

print(f"\n[PROTECTED] REGULATORY PROTECTION:")
print(f"   [SUCCESS] Rule version changes tracked immutably")
print(f"   [SUCCESS] Historical threshold reconstruction available")
print(f"   [SUCCESS] Decision rationale preserved for each transaction")
print(f"   [SUCCESS] No retroactive examination liability")

## Regulatory Compliance Validation

Validate that audit trails meet BSA/AML and FinCEN requirements:

In [ ]:
print("="*70)
print("⚖ REGULATORY COMPLIANCE VALIDATION")
print("="*70)

# Define required fields for BSA/AML compliance
required_aml_fields = [
    "regulation",
    "rule_version_history_preserved", 
    "mra_protection",
    "threshold_at_decision_time"
]

print("**Details:** Required BSA/AML Audit Trail Fields:")
for field in required_aml_fields:
    print(f"   • {field}")

if 'stored_id_1' in locals():
    sample_decision = db_backend.load_decision(stored_id_1)
    
    validation_result = backend.validate_regulatory_completeness(
        sample_decision,
        required_aml_fields
    )
    
    print(f"\n**Results:** COMPLIANCE STATUS: {'[SUCCESS] COMPLIANT' if validation_result['is_compliant'] else '[FAILED] NON-COMPLIANT'}")
    print(f"**Metrics:** Completeness Score: {validation_result['completeness_score']:.1%}")
    
    if validation_result['missing_fields']:
        print(f"[FAILED] Missing Fields: {', '.join(validation_result['missing_fields'])}")
    else:
        print("[SUCCESS] All required compliance fields present")
    
    print(f"\n**Achievement:** BSA/AML Examination Readiness: {'[SUCCESS] READY' if validation_result['is_compliant'] else '[FAILED] NOT READY'}")
    print(f"[PROTECTED] MRA Protection: {'[SUCCESS] ACTIVE' if validation_result['is_compliant'] else '[FAILED] INACTIVE'}")

else:
    print("\n[WARNING] No decisions available for validation")

## Value Summary for AML Monitoring

Summarize the key benefits of using Briefcase AI for AML transaction monitoring:

In [ ]:
print("="*70)
print("**Premium:** BRIEFCASE AI VALUE FOR AML MONITORING")
print("="*70)

benefits = [
    ("Complete rule version history preservation", "Never lose track of which rules were in effect"),
    ("Threshold-in-effect tracking", "Prove what thresholds applied to each decision"),
    ("SAR filing audit trail with rationale", "Document why SARs were filed or not filed"),
    ("MRA protection through audit completeness", "Avoid 'Matter Requiring Attention' citations"),
    ("OCC/FinCEN examination readiness", "Respond to examiner queries with confidence"),
    ("Retroactive review protection", "Defend historical decisions with evidence"),
    ("Rule change impact analysis", "Understand how policy changes affect operations")
]

for benefit, description in benefits:
    print(f"[SUCCESS] {benefit}")
    print(f"   → {description}")

# Summary statistics
decision_count = 0
rule_versions = set()

if 'stored_id_1' in locals():
    decision_count += 1
    rule_versions.add(aml_rules_v41['version'])

if 'stored_id_2' in locals():
    decision_count += 1
    rule_versions.add(aml_rules_v42['version'])

print(f"\n**Results:** SESSION SUMMARY:")
print(f"   Decisions Processed: {decision_count}")
print(f"   Rule Versions Tracked: {', '.join(sorted(rule_versions))}")
print(f"   Examination Readiness: [SUCCESS] Complete")
print(f"   Regulatory Compliance: [SUCCESS] BSA/AML Requirements Met")

## Summary & Key Accomplishments

### [SUCCESS] What We Accomplished

1. **Rule Version Tracking**: Demonstrated immutable preservation of AML rule versions and thresholds
2. **Threshold History**: Captured exact thresholds in effect at decision time for each transaction
3. **SAR Filing Audit**: Created complete audit trails for analyst decisions and SAR filings
4. **Rule Change Impact**: Showed how mid-year rule changes are properly documented and defended
5. **Examiner Readiness**: Demonstrated ability to respond to complex regulatory queries

### **Objective:** Key Regulatory Benefits

- **MRA Protection**: Complete audit trails prevent "Matter Requiring Attention" citations
- **BSA/AML Compliance**: All FinCEN requirements for decision documentation met
- **OCC Examination Support**: Ready responses to examiner questions about rule changes
- **Retroactive Review Defense**: Historical decisions justified with preserved context

### [ALERT] Critical Compliance Problem Solved

**The Problem**: When AML rules change mid-year, banks face retroactive examination liability where examiners question why similar transactions were treated differently before and after rule changes.

**The Solution**: Briefcase AI captures the exact rule version and threshold in effect at each decision point, creating an immutable audit trail that proves compliance decisions were made correctly based on rules in effect at the time.

### **Launch:** Production Implementation Guidance

1. **Integration**: Connect to your existing AML monitoring system (NICE Actimize, SAS, etc.)
2. **Rule Management**: Implement automated rule version tracking for all policy changes
3. **Analyst Workflow**: Integrate decision capture into analyst review processes
4. **Examiner Response**: Set up automated examiner query response capabilities
5. **Monitoring**: Implement alerts for rule change impacts and audit trail gaps

### **Reference:** Next Steps

- Integrate with your AML monitoring platform
- Customize rule version tracking for your specific policies
- Train analysts on audit trail importance
- Set up examiner response procedures
- Implement ongoing monitoring for compliance gaps

---

**[SECURED] Compliance Note**: This implementation demonstrates audit trail patterns for AML transaction monitoring compliance. Always validate specific BSA/AML requirements with qualified compliance and legal counsel for your jurisdiction and regulatory environment.

## Bitemporal Replay Demonstration

The earlier capture layer records *what* the system did. This section adds
the replay layer — proving *what was known* at decision time, so an auditor
can reconstruct the decision offline against the domain evidence as it
stood on the day of adjudication.

Primitives used here: `BitemporalRecord`, `InMemoryBitemporalStore`,
`append_correction`, `AsOfView`, `PolicyRegistry`, `ExaminerBundle`.

For each primitive in isolation, see [`patterns/`](../../patterns/).
For the same primitives composed into a cross-border payments narrative,
see [`agentic-payments/`](../../agentic-payments/).


### Seed a bitemporal KYC store

In [ ]:
# Self-contained imports (safe to re-run after the domain cells above).
import json
from datetime import datetime, timedelta, timezone

from briefcase.bitemporal import (
    AsOfView, BitemporalRecord, InMemoryBitemporalStore, append_correction,
)
from briefcase.compliance import BundleIntegrityError, ExaminerBundle
from briefcase.routing import (
    AgentRoutingDecision, PolicyRegistry, PolicyRule, PolicyVersion,
)

utc = timezone.utc
decision_time = datetime.now(utc) - timedelta(days=90)
correction_time = datetime.now(utc)

kyc = InMemoryBitemporalStore()
cp_kyc = BitemporalRecord.new(
    key="kyc:cp-88",
    valid_time=decision_time,
    value={"counterparty_id": "cp-88", "beneficial_owner": "J. Smith",
           "jurisdiction": "KY", "pep_flag": False, "screening_score": 0.12},
    source="internal_kyc",
    source_trust_level="primary",
    transaction_time=decision_time,
)
kyc.append(cp_kyc)
print(f"Seeded KYC store: cp-88 beneficial_owner='J. Smith'")

### Beneficial-owner restatement after EDD

Enhanced due diligence later reveals the ultimate beneficial owner is a PEP. The KYC record is restated; the original snapshot (what the alert engine saw) is preserved.

In [ ]:
append_correction(
    kyc, cp_kyc,
    corrected_value={"counterparty_id": "cp-88", "beneficial_owner": "H. Volkov (PEP)",
                     "jurisdiction": "KY", "pep_flag": True, "screening_score": 0.91,
                     "restatement_reason": "edd_disclosed_ultimate_owner"},
    transaction_time=correction_time,
)
print(f"Correction appended at {correction_time.date()}: pep_flag=True")

### Replay: live vs. as-of alert day

In [ ]:
live = kyc.latest("kyc:cp-88").value
print(f"Live view (today):    pep_flag={live['pep_flag']}  score={live['screening_score']}")
with AsOfView(kyc, transaction_time=decision_time) as view:
    replay = view.latest("kyc:cp-88").value
    print(f"As-of alert day:      pep_flag={replay['pep_flag']}  score={replay['screening_score']}")
print("\nReview 'why didn't you escalate?' — replay shows no PEP signal at alert time.")

### ExaminerBundle — content-addressed, verifiable, tamper-evident

In [ ]:
policy = PolicyVersion(
    policy_id="aml_monitoring",
    version="4.1.0",
    description="Alert on screening_score > 0.7 OR pep_flag.",
    rules=[PolicyRule(
        rule_id="threshold_alert",
        condition={"screening_score_gt": 0.7},
        choice="alert",
        rationale="Score threshold exceeded",
    )],
    default_choice="clear",
)
registry = PolicyRegistry()
registry.publish(policy, valid_from=decision_time, transaction_time=decision_time)

routing_decision = AgentRoutingDecision(
    decision_id="replay-demo",
    use_case="aml_monitoring",
    context={"counterparty": "cp-88", "notional_usd": 45_000},
    candidates=["clear", "alert", "sar_file"],
    selected="clear",
    policy_id="aml_monitoring",
    policy_version="4.1.0",
    matched_rule_id="threshold_alert",
    evidence_refs=[cp_kyc.record_id],
    rationale="screening_score 0.12 under 0.7 threshold; no PEP flag at decision time",
    decided_at=decision_time,
)

bundle = ExaminerBundle.build(
    routing_decision,
    evidence_store=kyc,
    policy_registry=registry,
    metadata={"regulation": "BSA/AML"},
)
print(f"content_hash:   {bundle.content_hash}")
print(f"evidence rows:  {len(bundle.evidence)}")

bundle.verify()
print("verify() on untouched bundle:    OK")

payload = bundle.to_json(indent=2)
ExaminerBundle.from_json(payload).verify()
print("verify() after JSON round-trip:  OK")

tampered = json.loads(payload)
tampered["decision"]["selected"] = "alert"
try:
    ExaminerBundle.from_dict(tampered).verify()
except BundleIntegrityError as e:
    print(f"verify() on tampered bundle:     REJECTED ({type(e).__name__})")